In [24]:
##1. Classifier: initialization & storage
pythonclass MyKNNClassifier:
    def __init__(self, n_neighbors=3, weights="uniform"):
        if not isinstance(n_neighbors, int) or n_neighbors <= 0:
            raise ValueError("n_neighbors must be a positive integer")
        if weights not in ("uniform", "distance"):
            raise ValueError("weights must be either 'uniform' or 'distance'")

        self.n_neighbors = n_neighbors
        self.weights = weights
        self.X_train = None
        self.y_train = None

##2. Classifier: fit (store training data)
python    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if len(X) != len(y):
            raise ValueError("X and y must have the same number of samples")
        if self.n_neighbors > len(X):
            raise ValueError("n_neighbors cannot be greater than number of training samples")

        self.X_train = X
        self.y_train = y
        return self

##3. Classifier: distance computation
python    def _euclidean_distances(self, x):
        """Vectorized Euclidean distance from one test sample to all training samples."""
        return np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))

##4. Classifier: single-sample prediction
python    def _predict_one(self, x):
        distances = self._euclidean_distances(x)
        neighbor_indices = np.argpartition(distances, self.n_neighbors - 1)[:self.n_neighbors]
        neighbor_labels = self.y_train[neighbor_indices]
        neighbor_distances = distances[neighbor_indices]

        if self.weights == "uniform":
            counts = Counter(neighbor_labels)
            return counts.most_common(1)[0][0]

        elif self.weights == "distance":
            class_weights = {}
            for label, dist in zip(neighbor_labels, neighbor_distances):
                if dist == 0:
                    return label
                weight = 1.0 / dist
                class_weights[label] = class_weights.get(label, 0.0) + weight
            return max(class_weights, key=class_weights.get)

##5. Classifier: batch prediction & accuracy
python    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(x) for x in X])

def score(self, X, y):
      y_pred = self.predict(X)
      return np.mean(y_pred == y)

##6. Utility: confusion matrix printer
pythondef print_confusion_matrix(cm, class_names, title):
    print(f"\n{title}")
    print("-" * len(title))
    print("Predicted ->")
    header = "Actual".ljust(12) + "".join(name.ljust(12) for name in class_names)
    print(header)
    for i, row in enumerate(cm):
        row_text = class_names[i].ljust(12) + "".join(str(val).ljust(12) for val in row)
        print(row_text)

##7. Experiment runner: data loading & preprocessing
pythondef run_experiment(n_neighbors=5, weights="uniform"):
    iris = load_iris()
    X, y = iris.data, iris.target
    target_names = iris.target_names

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

##8. Experiment runner: custom KNN evaluation
python    my_knn = MyKNNClassifier(n_neighbors=n_neighbors, weights=weights)

    start_fit = time.perf_counter()
    my_knn.fit(X_train_scaled, y_train)
    my_fit_time = time.perf_counter() - start_fit

    start_pred = time.perf_counter()
    my_pred = my_knn.predict(X_test_scaled)
    my_pred_time = time.perf_counter() - start_pred

    my_acc = accuracy_score(y_test, my_pred)
    my_cm = confusion_matrix(y_test, my_pred)

    print("\nMyKNNClassifier Results")
    print(f"Accuracy: {my_acc:.4f}")
    print(classification_report(y_test, my_pred, target_names=target_names))
    print_confusion_matrix(my_cm, target_names, "Confusion Matrix (MyKNN)")

##9. Experiment runner: sklearn KNN evaluation
python    sk_knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights)

    start_fit = time.perf_counter()
    sk_knn.fit(X_train_scaled, y_train)
    sk_fit_time = time.perf_counter() - start_fit

    start_pred = time.perf_counter()
    sk_pred = sk_knn.predict(X_test_scaled)
    sk_pred_time = time.perf_counter() - start_pred

    sk_acc = accuracy_score(y_test, sk_pred)
    sk_cm = confusion_matrix(y_test, sk_pred)

    print("\nSklearn KNN Results")
    print(f"Accuracy: {sk_acc:.4f}")
    print(classification_report(y_test, sk_pred, target_names=target_names))
    print_confusion_matrix(sk_cm, target_names, "Confusion Matrix (sklearn KNN)")

##10. Experiment runner: comparison summary & return
python    print(f"\n{'Metric':<20}{'MyKNN':<18}{'sklearn KNN':<18}")
    print("-" * 56)
    print(f"{'Accuracy':<20}{my_acc:<18.4f}{sk_acc:<18.4f}")
    print(f"{'Train time (s)':<20}{my_fit_time:<18.8f}{sk_fit_time:<18.8f}")
    print(f"{'Predict time (s)':<20}{my_pred_time:<18.8f}{sk_pred_time:<18.8f}")

    return {
        "n_neighbors": n_neighbors, "weights": weights,
        "my_accuracy": my_acc, "sk_accuracy": sk_acc,
        "my_train_time": my_fit_time, "sk_train_time": sk_fit_time,
        "my_predict_time": my_pred_time, "sk_predict_time": sk_pred_time
    }

##11. Complexity analysis printer
pythondef print_complexity_analysis():
    print("MyKNNClassifier")
    print("  Train:   O(n*d) — lazy learner, just stores data")
    print("  Predict: O(n*d) — distances to all training samples")
    print("  Space:   O(n*d)")
    print()
    print("sklearn KNeighborsClassifier")
    print("  Same algorithm but optimized low-level code — faster in practice")

##12. Entry point: run all experiments & report best result
pythonif __name__ == "__main__":
    results = []
    results.append(run_experiment(n_neighbors=3, weights="uniform"))
    results.append(run_experiment(n_neighbors=5, weights="uniform"))
    results.append(run_experiment(n_neighbors=5, weights="distance"))

    print_complexity_analysis()

    best = max(results, key=lambda x: x["my_accuracy"])
    print(
        f"\nBest result: n_neighbors={best['n_neighbors']}, "
        f"weights='{best['weights']}', accuracy={best['my_accuracy']:.4f}"
    )
    print("Custom KNN matched sklearn accuracy in all cases; sklearn was faster in prediction.") Sonnet 4.6

SyntaxError: invalid syntax (3935801080.py, line 2)